Roughness - need higher order terms.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import sys
import shutil
import itertools as it
import contextlib
from scipy.optimize import minimize
from sklearn.preprocessing import MinMaxScaler
import statsmodels.formula.api as smf
import json

sys.path.append("/Users/octaviacrompton/Projects/roughness-scale/swof_code")

In [ ]:
my_modules = ['plot_SWOF', "read_SWOF", "write_SWOF", 
              "plot_config", "topo", "source_functions_1p3"]

for mod in my_modules:
    if mod in sys.modules: 
        del sys.modules[mod] 

from plot_SWOF import *
from read_SWOF import *
from plot_config import *
from topo import *
from write_SWOF import *
from source_functions_1p3 import *

In [ ]:
project_dir = "/Users/octaviacrompton/Dropbox/FullCSWOF/Tests/"
cases = [d for d in os.listdir(project_dir) if 
            ('DS_Store' not in d) and ('figures' not in d)]


## Load simulations

In [ ]:
def load_output(out_dir, load = 1):
    path = os.path.join(out_dir, "sim_list.json")

    # read_summary(sim_list[0], default, out_dir)
    with open(path, "r") as read_file:
        sim_list = json.load(read_file)

    path = os.path.join(out_dir, "default.json")

    with open(path, "r") as read_file:
        default = json.load(read_file)   

    if load == 1:
        summary, badlist = load_summary(sim_list, default, out_dir)    
    else:
        summary, badlist = get_summary(sim_list, default, out_dir)
    
    return summary, badlist, sim_list




In [ ]:
out_dir = os.path.join(project_dir, "runaround3")
summary, badlist, sim_list = load_output(out_dir, load = 1)

# out_dir = os.path.join(project_dir, "runaround_rough")
# summary, badlist, sim_list = load_output(out_dir, load = 1)

In [ ]:
for key in summary.index:
    sim = summary.loc[key]
    summary.loc[key, 'fV'] = (sim.veg.mean())

In [ ]:
summary.fV.hist()

In [ ]:

print_input_params(sim_list)

In [ ]:
plot_hydrographs(summary.query("tr == 60 and l == 400 and p == 5"))

In [ ]:
plot_hydrographs(summary.query("tr == 20 and l == 50 and p == 8 "))

In [ ]:

out_dir_eq = os.path.join(project_dir, "runaround_equiv3")
summary_equiv, badlist_equiv, sim_list_equiv = load_output(out_dir_eq, load = 1)


In [ ]:
print_input_params(sim_list_equiv)

In [ ]:
summary = add_terms(summary)
summary = add_lengthscales(summary)

In [ ]:
summary = add_Reynolds_numbers(summary)
summary = add_Re_all(summary)

In [ ]:
summary_equiv = add_terms(summary_equiv)
summary_equiv = add_lengthscales(summary_equiv)

In [ ]:
summary = insert_fld(summary, 'equiv_hydro')
summary = insert_fld(summary, 'equiv_hydro_IF')

In [ ]:
from scipy.stats import gmean

In [ ]:
for key in summary.index:
    
    sim  = summary.loc[key]
    subset = summary_equiv.query("l == {0} and p == {1} and tr == {2} ".format(sim.l, sim.p, sim.tr))
    
    errors = [np.mean((sim.hydro[:len(hydro)] - hydro)**2)**0.5 for hydro in subset.hydro]
    ind = np.where(errors == np.min(errors))[0][0]

    if ind == 0:
        summary.loc[key, 'boundary'] = -1 
    elif ind == len(errors):
        summary.loc[key, 'boundary'] = 1 
    else:
        summary.loc[key, 'boundary'] = 0 
        
    r_equiv = subset.iloc[ind].alpha_b
    summary.loc[key, 'r_equiv'] = r_equiv 
    summary.loc[key, 'IF_err'] = np.abs(sim.IF - subset.iloc[ind].IF)
    
    
    summary.at[key, 'equiv_hydro'] = list(subset.query("alpha_b == {0}".format(r_equiv)).iloc[0].hydro)
    assert(len(subset.query("alpha_b == {0}".format(r_equiv))) > 0)
    
    a = sim.veg
    a[a==1] = sim.alpha_v
    a[a==0] = sim.alpha_b

    summary.loc[key, 'r_avg'] = np.mean(a)
    summary.loc[key, 'r_gmean'] = gmean(gmean(a))
    
    hhh = sim.hc.copy()
    hhh[hhh < 0.002] = np.nan
    summary.loc[key, 'Cd'] = 9.8*np.nanmean(hhh**(-1/3.)*a**0.5)

    x, y = make_grid(sim)
    z = rescale(1-y)*sim.So*sim.l    
    
    summary.loc[key, 'z_std'] = np.std((sim.zc - z)**2)**0.5
    
    summary.loc[key, 'hydro_err'] = np.min(errors)/sim.hydro.max()

    r_equiv = subset.iloc[np.where(errors < np.min(errors)*1.02)[0]].alpha_b.mean()
    summary.loc[key, 'r_equiv5'] = r_equiv 
    r_std =     r_equiv = subset.iloc[np.where(errors < np.min(errors)*1.05)[0]].alpha_b.std()
    summary.loc[key, 'r_std'] = r_std 
    
    i_tr = sim.i_tr
    errors = [np.mean((sim.hydro[:i_tr] - hydro[:i_tr])**2)**0.5 for hydro in subset.hydro]
    r_equiv = subset.iloc[np.where(errors == np.min(errors))[0][0]].alpha_b
    summary.loc[key, 'r_equiv_rise'] = r_equiv 
    
    i_tr = sim.i_tr
    errors = [np.mean((sim.hydro[i_tr:len(hydro)] - hydro[i_tr:])**2)**0.5 for hydro in subset.hydro]
    r_equiv = subset.iloc[np.where(errors == np.min(errors))[0][0]].alpha_b
    summary.loc[key, 'r_equiv_fall'] = r_equiv 
    
    errors = [np.mean((sim.IF - IF)**2)**0.5 for IF in subset.IF]    
    r_equiv = subset.iloc[np.where(errors == np.min(errors))[0][0]].alpha_b
    summary.loc[key, 'r_equiv_IF'] = r_equiv 
    summary.at[key, 'equiv_hydro_IF'] = list(subset.query("alpha_b == {0}".format(r_equiv)).iloc[0].hydro)
    
summary['fV'] = [veg.mean().round(3) for veg in summary.veg]
summary['fV'] = summary['fV'].astype(float).round(3)
summary['fV'] = [np.round(a,3) for a in summary.fV]
summary['fVo'] = [float(name.split(',')[2].split("-")[1]) for name in summary.index]    

In [ ]:
def integrate_U_across(r_2d, m, So, p, dx=2):
    """
    Compute ∫ ((p x)^m / r(x) S_o^w)^(1/(m+1)) dx along axis=1 of a 2D array.

    Parameters:
    - r_2d: 2D numpy array (x along axis=1)
    - m: exponent in the formula
    - So: slope term
    - p: scaling constant
    - dx: spacing along the x axis

    Returns:
    - 1D numpy array of integrals along axis=1 (one per row)
    """
    x = np.arange(r_2d.shape[1]) * dx / 3.6e5
    x_matrix = np.expand_dims(x, axis=0) * p  # shape (1, n) to broadcast across rows

    with np.errstate(divide='ignore', invalid='ignore'):
        expr = (x_matrix ** m / r_2d) ** (1 / (m + 1))
        expr[np.isnan(expr)] = 0
        expr[np.isinf(expr)] = 0

    integral = np.trapz(expr, dx=dx, axis=1)
    return integral * So ** (0.5 / (m + 1))


def integrate_h_across(r_2d, m, So, p, dx=2):
    """
    Compute ∫ ((x * r(x) * p * S_o^-w)^(1/(m+1)) dx along axis=1 of a 2D array.

    Parameters:
    - r_2d: 2D numpy array (x along axis=1)
    - m: exponent in the formula
    - So: slope term
    - p: scaling constant
    - dx: spacing along the x axis

    Returns:
    - 1D numpy array of integrals along axis=1 (one per row)
    """
    x = np.arange(r_2d.shape[1]) * dx / 3.6e5

    x_matrix = np.expand_dims(x, axis=0) * p  # shape (1, n) to broadcast across rows

    with np.errstate(divide='ignore', invalid='ignore'):
        expr = (x_matrix * r_2d) ** (1 / (m + 1.))
        expr[np.isnan(expr)] = 0
        expr[np.isinf(expr)] = 0

    integral = np.trapz(expr, dx=dx, axis=1)
    return integral * So ** (-0.5 / (m + 1.))


In [ ]:
sim.alpha_v, sim.alpha_b

In [ ]:
# for key in summary.index:
#     sim  = summary.loc[key]   
    
#     n = sim.veg
#     n[n==1] = sim.alpha_v
#     n[n==0] = sim.alpha_b

#     K = sim.veg.astype(float).copy()
#     K[K==1] = sim.Ks_v
#     K[K==0] = sim.Ks_v # since Ks_b = 'Ks_v'
    
#     # grab the time slice and ensure float arrays
#     h_t  = np.asarray(sim.hc, dtype=float)[int(sim.i_tr)]   # shape (ny, nx)
#     #h_t[h_t < 0.01] = np.nan

#     U_t  = np.asarray(sim.vc, dtype=float)[int(sim.i_tr)]   # shape (ny, nx)
#     #U_t[U_t < 0.001] = np.nan        
    
#     dx   = float(sim.dx)

#     # central difference in x at interior points → shape (ny, nx-2)
#     dUdx = (U_t[:, 2:] - U_t[:, :-2]) / (2 * dx)

#     # match U at the same x-locations (midpoints)
#     U_mid = U_t[:, 1:-1]                                    # shape (ny, nx-2)


#     summary.loc[key, 'n_misc'] = np.nanmean(h_t**(1/6)*n**(1/2))    
#     summary.loc[key, 'n_tr'] = np.nanmean(h_t**(2/3))*sim.So**(1/2)/np.nanmean(U_t) 
#     summary.loc[key, 'Cd'] = 9.8*np.nanmean(h_t)*sim.So/np.nanmean(U_t**2)        
    
#     # from correction factor stud`
#     summary.loc[key, '<U>_correct'] = 5/7.*np.nanmean( 1/n*((sim.p - sim.Ks_v)/3.6e5*sim.l)**(2/3)*sim.So**.5)**(3/5)
#     summary.loc[key, '<h>_correct'] = 5/8.*np.nanmean( ((sim.p - sim.Ks_v )/3.6e5*sim.l*n/sim.So**.5))**(3/5)        

#     Va = integrate_U_across(n, .66, sim.So, sim.p - sim.Ks_v, dx=2)
#     summary.loc[key, '<Ua>'] =  np.nanmean(Va/sim.l)
#     ha = integrate_h_across(n, .66, sim.So, sim.p - sim.Ks_v, dx=2)
#     summary.loc[key, '<ha>'] =  np.nanmean(ha/sim.l)  
#     summary.loc[key, '<na>'] = np.nanmean(ha/sim.l)**(2/3)/np.nanmean(Va/sim.l)*sim.So**0.5                
    
#     summary.loc[key, '<U>'] = np.nanmean(U_t)
#     summary.loc[key, '<h>'] = np.nanmean(h_t)
#     summary.loc[key, '<U>/<h>'] =np.nanmean(U_t)/np.nanmean(h_t)    

#     summary.loc[key, 'n_gmean'] = gmean(np.ravel(n))      # geometric mean of prescribed manning's n
#     summary.loc[key, '<n>'] = np.nanmean(n)        # arithmetic mean of prescribed manning's n
#     summary.loc[key, '<n2>'] = np.nanmean(n**2)
#     summary.loc[key, '<n>2'] = np.nanmean(n)**2    
    
#     summary.loc[key, '<Sf>'] = np.nanmean(n**2*(h_t**(-4/3))*(U_t**2))        
    
#     # <Sf> = <U>^2 <n^2> <h>^{-4/3} 
#     # summary.loc[key, '<Sf>'] = np.nanmean(a**2)*(np.nanmean(h_t))**(-4/3)*(np.nanmean(U_t)**2)            
#     #`summary.loc[key, '<Sf_gmean>'] = gmean(a.ravel()**2)*(np.nanmean(h_t))**(-4/3)*(np.nanmean(U_t)**2)            
    
#     # <Sf alt> = <U>^2 <n>^2 <h>^(-4/3)    
#     #summary.loc[key, '<Sf>_alt'] = np.nanmean(a)**2*(np.nanmean(h_t))**(-4/3)*(np.nanmean(U_t)**2)
 
#     # Means & fluctuations
#     U_bar = np.nanmean(U_t)          # <U>
#     h_bar = np.nanmean(h_t)          # <h>z
#     n_bar = np.nanmean(a)                 # <n>
    
#     Up  = U_t - U_bar                    # U'
#     hp  = h_t - h_bar                    # h'
#     np_ = n    - n_bar                    # n'
    
    
#     dUdx = ((sim.vc[sim.i_tr, :, 2:] -  sim.vc[sim.i_tr, :, :-2])/2/sim.dx)    
#     dUp_dx = (Up[:, 2:] -  Up[:, :-2])/2/sim.dx
    
#     Um = np.nanmean(np.asarray(sim.vc), axis=(1, 2))
#     summary.loc[key, 'd<U>/dt'] = np.gradient(Um, float(sim['dt']))[int(sim.i_tr)]

#     summary.loc[key, '<dU/dx>'] =  np.nanmean(dUdx)
    
#     # ⟨U dU/dx⟩ : mean of the pointwise product
#     summary.loc[key, '<U dU/dx>'] = np.nanmean(U_mid * dUdx)

#     # ⟨U⟩⟨dU/dx⟩ : product of the means
#     summary.loc[key, '<U><dU/dx>'] = np.nanmean(U_mid) * np.nanmean(dUdx)
        

#     summary.loc[key, '<Up dUp/dx>']  = np.nanmean(Up[:, 1:-1] * dUp_dx)
#     summary.loc[key, '<Up2>']  = np.nanmean(Up**2)
    
#     # depth gradients using h_t/hp with same stencil
#     dh_dx  = (h_t[:, 2:] - h_t[:, :-2]) / (2 * dx)
#     dhp_dx = (hp[:, 2:]  - hp[:, :-2]) / (2 * dx)
#     summary.loc[key, '<dh/dx>']  = np.nanmean(dh_dx)
#     summary.loc[key, '<dhp/dx>'] = np.nanmean(dhp_dx)
   

#     # Prefactor: <n^2><h>^-4/3  (you can reuse '<a2>' if already computed)
#     summary.loc[key, '<n2><h>^-4/3'] = np.nanmean(n**2) * (h_bar ** (-4/3))


#     # supporting second-moment stats
#     summary.loc[key, '<Up2>']   = np.nanmean(Up**2)                # <U'^2>
#     summary.loc[key, '<np2>']   = np.nanmean(np_**2)               # <n'^2>
#     summary.loc[key, '<npUp>']  = np.nanmean(np_ * Up)             # <n' U'>
#     summary.loc[key, '<Up hp>'] = np.nanmean(Up * hp)              # <U' h'>  
#     summary.loc[key, '<np hp>'] = np.nanmean(np_ * hp)             # <n' h'>

#     # prefactors from the linearized h^{-4/3}
#     h_pref     = h_bar**(-4/3)        # <h>^{-4/3}
#     h_pref_lin = h_bar**(-7/3)        # <h>^{-4/3} / <h> = <h>^{-7/3}

    
#     # ----- O(ε^0) + O(ε^2) terms (inside the first bracket) -----
#     summary.loc[key, '<Sf>_nbar2_Ubar2'] = h_pref * (n_bar**2) * (U_bar**2)
#     summary.loc[key, '<Sf>_nbar2_Up2']   = h_pref * (n_bar**2) * summary.loc[key, '<Up2>']
#     summary.loc[key, '<Sf>_Ubar2_np2']   = h_pref * (U_bar**2) * summary.loc[key, '<np2>']
#     summary.loc[key, '<Sf>_cross_nU']    = h_pref * (4. * n_bar * U_bar * summary.loc[key, '<npUp>'])

#     # ----- linearized h' terms (the -8/3 * ... pieces) -----
#     summary.loc[key, '<Sf>_C_Uhp_lin'] = -(8.0/3.0) * h_pref_lin * (n_bar**2) * U_bar * summary.loc[key, '<Up hp>']
#     summary.loc[key, '<Sf>_C_nhp_lin'] = -(8.0/3.0) * h_pref_lin * (U_bar**2) * n_bar * summary.loc[key, '<np hp>']

#     # ----- total (this is your boxed approximation) -----
#     summary.loc[key, '<Sf>_approx2'] = (
#         summary.loc[key, '<Sf>_nbar2_Ubar2'] +
#         summary.loc[key, '<Sf>_nbar2_Up2']   +
#         summary.loc[key, '<Sf>_Ubar2_np2']   +
#         summary.loc[key, '<Sf>_cross_nU']    +
#         summary.loc[key, '<Sf>_C_Uhp_lin']   +
#         summary.loc[key, '<Sf>_C_nhp_lin']
#     )

#     # (optional) compare to your direct mean if available
#     # If you store the exact mean as '<Sf>_direct' (or 'Sf' in older code), compute a residual:
# #     if '<Sf>_direct' in summary.columns:
# #         summary.loc[key, '<Sf>_resid(approx2-direct)'] = summary.loc[key, '<Sf>_approx2'] - summary.loc[key, '<Sf>_direct']
# #     elif 'Sf' in summary.columns:
# #         summary.loc[key, '<Sf>_resid(approx2-direct)'] = summary.loc[key, '<Sf>_approx2'] - summary.loc[key, 'Sf']


   
#     # Distributed components (each already multipl ied by the prefactor)
#     summary.loc[key, '<Sf>_U2']  = summary.loc[key, '<n2><h>^-4/3'] * (U_bar**2)
#     summary.loc[key, '<Sf>_Up2'] = summary.loc[key, '<n2><h>^-4/3'] * np.nanmean(Up**2)
#     summary.loc[key, '<Sf>_C1']  = summary.loc[key, '<n2><h>^-4/3'] * (- 8/3 * U_bar * np.nanmean(Up*hp) / h_bar)
#     summary.loc[key, '<Sf>_C2']  = summary.loc[key, '<n2><h>^-4/3'] * (- 4/3 * np.nanmean(Up**2*hp) / h_bar)

#     summary.loc[key, '<Sf>_first']  = summary.loc[key, '<n2><h>^-4/3'] * (U_bar**2  + np.nanmean(Up**2))
    
    
#     # Sum (the approximation to <Sf>)
#     summary.loc[key, '<Sf>_approx'] = (
#         summary.loc[key, '<Sf>_U2'] +
#         summary.loc[key, '<Sf>_Up2'] +
#         summary.loc[key, '<Sf>_C1'] +
#         summary.loc[key, '<Sf>_C2']
#     )
    

In [ ]:
# for key in summary.index:
    
#     sim = summary.loc[key]

#     # ---- indices & constants
#     i_tr = int(sim.i_tr)
#     dx   = float(sim.dx)
#     g    = 9.81

#     # ---- Manning n field (float, safe copy)
#     veg = np.asarray(sim.veg)
#     n = veg.astype(float).copy()
#     n[veg == 1] = float(sim.alpha_v)
#     n[veg == 0] = float(sim.alpha_b)

#     # ---- time-slice arrays (float)
#     h_t = np.asarray(sim.hc, dtype=float)[i_tr]  # (ny, nx)
#     U_t = np.asarray(sim.vc, dtype=float)[i_tr]  # (ny, nx)

#     # ---- central difference (aligns with midpoints)
#     dUdx  = (U_t[:, 2:] - U_t[:, :-2]) / (2 * dx)   # (ny, nx-2)
#     U_mid = U_t[:, 1:-1]                              # (ny, nx-2)

#     # ---- diagnostics
#     summary.loc[key, 'n_misc'] = np.nanmean(h_t**(1/6) * n**0.5)
#     summary.loc[key, 'n_tr']   = np.nanmean(h_t**(2/3)) * (sim.So**0.5) / np.nanmean(U_t)
#     summary.loc[key, 'Cd']     = g * np.nanmean(h_t) * sim.So / np.nanmean(U_t**2)

#     # ---- correction-factor study
#     # n_safe = np.where(n > 0, n, np.nan)
#     summary.loc[key, '<U>_correct'] = (5/7) * np.nanmean(
#         (1/n) * ((sim.p - sim.Ks_v)/3.6e5 * sim.l)**(2/3) * sim.So**0.5)**(3/5)
#     summary.loc[key, '<h>_correct'] = (5/8) * np.nanmean(
#         ((sim.p - sim.Ks_v)/3.6e5 * sim.l * n / sim.So**0.5))**(3/5)

#     # ---- cross-section integrals (use sim.dx)
#     try:
#         Va = integrate_U_across(n, 0.66, sim.So, sim.p - sim.Ks_v, dx=dx)
#         ha = integrate_h_across(n, 0.66, sim.So, sim.p - sim.Ks_v, dx=dx)
#         summary.loc[key, '<Ua>'] = np.nanmean(Va / sim.l)
#         summary.loc[key, '<ha>'] = np.nanmean(ha / sim.l)
#         summary.loc[key, '<na>'] = (np.nanmean(ha / sim.l))**(2/3) / np.nanmean(Va / sim.l) * sim.So**0.5
#     except NameError:
#         summary.loc[key, '<Ua>'] = np.nan
#         summary.loc[key, '<ha>'] = np.nan
#         summary.loc[key, '<na>'] = np.nan

#     # ---- means
#     U_bar = float(np.nanmean(U_t))
#     h_bar = float(np.nanmean(h_t))
#     n_bar = float(np.nanmean(n))

#     summary.loc[key, '<n>']     = n_bar    
#     summary.loc[key, '<U>']     = U_bar
#     summary.loc[key, '<h>']     = h_bar
#     summary.loc[key, '<U>/<h>'] = U_bar / h_bar
#     summary.loc[key, 'n_gmean'] = gmean(n.ravel())

#     summary.loc[key, '<n2>']    = float(np.nanmean(n**2))
#     summary.loc[key, '<n>2']    = n_bar**2

#     # ---- direct mean friction slope
#     # <Sf> = <U>^2 <n^2> <h>^(-4/3)
#     Sf_direct = float(np.nanmean(n**2 * h_t**(-4/3) * U_t**2))
#     summary.loc[key, '<Sf>']        = Sf_direct
#     summary.loc[key, '<Sf>_direct'] = Sf_direct  # convenient alias

#     # ---- fluctuations
#     Up = U_t - U_bar
#     hp = h_t - h_bar
#     np_ = n - n_bar

#     dUp_dx = (Up[:, 2:] - Up[:, :-2]) / (2 * dx)

#     # time derivative of <U>
#     Um = np.nanmean(np.asarray(sim.vc), axis=(1, 2))
#     summary.loc[key, 'd<U>/dt'] = np.gradient(Um, float(sim['dt']))[i_tr]

#     # velocity gradient summaries
#     summary.loc[key, '<dU/dx>']     = float(np.nanmean(dUdx))
#     summary.loc[key, '<U dU/dx>']   = float(np.nanmean(U_mid * dUdx))
#     summary.loc[key, '<U><dU/dx>']  = float(np.nanmean(U_mid) * np.nanmean(dUdx))
#     summary.loc[key, '<Up dUp/dx>'] = float(np.nanmean(Up[:, 1:-1] * dUp_dx))
#     summary.loc[key, '<Up2>']       = float(np.nanmean(Up**2))

#     summary.loc[key, '<hp2>']       = float(np.nanmean(hp**2))    


#     # depth gradients (same stencil)
#     dh_dx  = (h_t[:, 2:] - h_t[:, :-2]) / (2 * dx)
#     dhp_dx = (hp[:, 2:]  - hp[:, :-2]) / (2 * dx)
#     summary.loc[key, '<dh/dx>']  = float(np.nanmean(dh_dx))
#     summary.loc[key, '<dhp/dx>'] = float(np.nanmean(dhp_dx))

#     # ---- prefactor note: <n2><h>^-4/3
#     summary.loc[key, '<n2><h>^-4/3'] = np.nanmean(n**2) * (h_bar ** (-4/3))

#     # ---- second-moment stats for decomposition
#     summary.loc[key, '<np2>']   = float(np.nanmean(np_**2))
#     summary.loc[key, '<npUp>']  = float(np.nanmean(np_ * Up))
#     summary.loc[key, '<Up hp>'] = float(np.nanmean(Up * hp))
#     summary.loc[key, '<np hp>'] = float(np.nanmean(np_ * hp))

#     # ---- h power factors
# #     h_pref     = h_bar**(-4/3)   # <h>^{-4/3}
# #     h_pref_lin = h_bar**(-7/3)   # <h>^{-7/3}

# #     # O(ε^0)+O(ε^2)
# #     summary.loc[key, '<Sf>_nbar2_Ubar2'] = h_pref * (n_bar**2) * (U_bar**2)
# #     summary.loc[key, '<Sf>_nbar2_Up2']   = h_pref * (n_bar**2) * summary.loc[key, '<Up2>']
# #     summary.loc[key, '<Sf>_Ubar2_np2']   = h_pref * (U_bar**2) * summary.loc[key, '<np2>']
# #     summary.loc[key, '<Sf>_cross_nU']    = h_pref * (4 * n_bar * U_bar * summary.loc[key, '<npUp>'])

# #     # linearized h' terms
# #     summary.loc[key, '<Sf>_C_Uhp_lin'] = -(8/3) * h_pref_lin * (n_bar**2) * U_bar * summary.loc[key, '<Up hp>']
# #     summary.loc[key, '<Sf>_C_nhp_lin'] = -(8/3) * h_pref_lin * (U_bar**2) * n_bar * summary.loc[key, '<np hp>']

# #     # totals
# #     summary.loc[key, '<Sf>_approx']  = (
# #         summary.loc[key, '<Sf>_nbar2_Ubar2'] +
# #         summary.loc[key, '<Sf>_nbar2_Up2']   +
# #         summary.loc[key, '<Sf>_Ubar2_np2']   +
# #         summary.loc[key, '<Sf>_cross_nU']
# #     )

# #     summary.loc[key, '<Sf>_approx2'] = (
# #         summary.loc[key, '<Sf>_approx']      +  
# #         summary.loc[key, '<Sf>_C_Uhp_lin']   +
# #         summary.loc[key, '<Sf>_C_nhp_lin']
# #     )

#     # ---- h power factors (prefactors for <h> exponents)
#     h_pref0 = h_bar**(-4/3)   # <h>^{-4/3}
#     h_pref1 = h_bar**(-7/3)   # <h>^{-7/3}
#     h_pref2 = h_bar**(-10/3)  # <h>^{-10/3}
#     h_pref3 = h_bar**(-13/3)  # <h>^{-13/3}
#     h_pref4 = h_bar**(-16/3)  # <h>^{-16/3}

#     # ---- binomial coefficients for (h_bar + h')^{-4/3}
#     c0 = 1.0
#     c1 = -4.0/3.0
#     c2 = 14.0/9.0
#     c3 = -140.0/81.0
#     c4 = 455.0/243.0

#     # ---- useful powers and mixed moments
#     Up2  = np.nanmean(Up**2)
#     np2  = np.nanmean(np_**2)
#     Uhp  = np.nanmean(Up * hp)
#     nhp  = np.nanmean(np_ * hp)

#     hp2  = np.nanmean(hp**2)
#     hp3  = np.nanmean(hp**3)
#     hp4  = np.nanmean(hp**4)

#     # third-/fourth-order moments needed
#     nUp2      = np.nanmean(np_ * (Up**2))                 # <n' U'^2>
#     n2Up      = np.nanmean((np_**2) * Up)                 # <n'^2 U'>
#     n2Up2     = np.nanmean((np_**2) * (Up**2))            # <n'^2 U'^2>

#     Up2hp_    = np.nanmean((Up**2) * hp)                  # <U'^2 h'>
#     n2hp_     = np.nanmean((np_**2) * hp)                 # <n'^2 h'>
#     nUhp      = np.nanmean(np_ * Up * hp)                 # <n' U' h'>
#     nU2hp     = np.nanmean(np_ * (Up**2) * hp)            # <n' U'^2 h'>
#     n2Uphp    = np.nanmean((np_**2) * Up * hp)            # <n'^2 U' h'>

#     Uph2      = np.nanmean(Up * (hp**2))                  # <U' (h')^2>
#     nh2       = np.nanmean(np_ * (hp**2))                 # <n' (h')^2>
#     U2h2      = np.nanmean((Up**2) * (hp**2))             # <U'^2 (h')^2>
#     nUh2      = np.nanmean(np_ * Up * (hp**2))            # <n' U' (h')^2>
#     n2h2      = np.nanmean((np_**2) * (hp**2))            # <n'^2 (h')^2>

#     Uh3       = np.nanmean(Up * (hp**3))                  # <U' (h')^3>
#     nh3       = np.nanmean(np_ * (hp**3))                 # <n' (h')^3>

#     # ---- T0 .. T4 bundles (as derived)
#     # T0 = <n^2 U^2>
#     T0 = (
#         (n_bar**2)*(U_bar**2)                # baseline
#         + (n_bar**2)*Up2
#         + (U_bar**2)*np2
#         + 4.0*n_bar*U_bar*np.nanmean(np_ * Up)
#         + 2.0*n_bar*nUp2
#         + 2.0*U_bar*n2Up
#         + n2Up2
#     )

#     # T1 = <n^2 U^2 h'>
#     T1 = (
#         2.0*(n_bar**2)*U_bar*Uhp
#         + 2.0*n_bar*(U_bar**2)*nhp
#         + (n_bar**2)*Up2hp_
#         + 4.0*n_bar*U_bar*nUhp
#         + (U_bar**2)*n2hp_
#         + 2.0*n_bar*nU2hp
#         + 2.0*U_bar*n2Uphp
#     )

#     # T2 = <n^2 U^2 (h')^2>
#     T2 = (
#         (n_bar**2)*(U_bar**2)*hp2
#         + 2.0*(n_bar**2)*U_bar*Uph2
#         + 2.0*n_bar*(U_bar**2)*nh2
#         + (n_bar**2)*U2h2
#         + 4.0*n_bar*U_bar*nUh2
#         + (U_bar**2)*n2h2
#     )

#     # T3 = <n^2 U^2 (h')^3>
#     T3 = (
#         (n_bar**2)*(U_bar**2)*hp3
#         + 2.0*(n_bar**2)*U_bar*Uh3
#         + 2.0*n_bar*(U_bar**2)*nh3
#     )

#     # T4 = <n^2 U^2 (h')^4>
#     T4 = (n_bar**2)*(U_bar**2)*hp4

#     # ---- assemble contributions with prefactors
#     Sf_T0 = h_pref0 * T0
#     Sf_T1 = c1 * h_pref1 * T1
#     Sf_T2 = c2 * h_pref2 * T2
#     Sf_T3 = c3 * h_pref3 * T3
#     Sf_T4 = c4 * h_pref4 * T4

#     # store detailed diagnostics (optional but handy)
#     summary.loc[key, '<Sf>_T0'] = float(Sf_T0)
#     summary.loc[key, '<Sf>_T1'] = float(Sf_T1)
#     summary.loc[key, '<Sf>_T2'] = float(Sf_T2)
#     summary.loc[key, '<Sf>_T3'] = float(Sf_T3)
#     summary.loc[key, '<Sf>_T4'] = float(Sf_T4)

#     # component-level breakdowns (some already exist; keep names consistent)
#     summary.loc[key, '<Sf>_T0_base']         = h_pref0 * (n_bar**2)*(U_bar**2)
#     summary.loc[key, '<Sf>_T0_Up2']          = h_pref0 * (n_bar**2)*Up2
#     summary.loc[key, '<Sf>_T0_np2']          = h_pref0 * (U_bar**2)*np2
#     summary.loc[key, '<Sf>_T0_cross_nU']     = h_pref0 * (4.0*n_bar*U_bar*np.nanmean(np_ * Up))
#     summary.loc[key, '<Sf>_T0_skew_nU2']     = h_pref0 * (2.0*n_bar*nUp2)
#     summary.loc[key, '<Sf>_T0_skew_n2U']     = h_pref0 * (2.0*U_bar*n2Up)
#     summary.loc[key, '<Sf>_T0_kurt']         = h_pref0 * n2Up2

#     summary.loc[key, '<Sf>_T1_Uhp']          = c1*h_pref1 * (2.0*(n_bar**2)*U_bar*Uhp)
#     summary.loc[key, '<Sf>_T1_nhp']          = c1*h_pref1 * (2.0*n_bar*(U_bar**2)*nhp)
#     summary.loc[key, '<Sf>_T1_Up2hp']        = c1*h_pref1 * ((n_bar**2)*Up2hp_)
#     summary.loc[key, '<Sf>_T1_n2hp']         = c1*h_pref1 * ((U_bar**2)*n2hp_)
#     summary.loc[key, '<Sf>_T1_nUhp']         = c1*h_pref1 * (4.0*n_bar*U_bar*nUhp)
#     summary.loc[key, '<Sf>_T1_nU2hp']        = c1*h_pref1 * (2.0*n_bar*nU2hp)
#     summary.loc[key, '<Sf>_T1_n2Uhp']        = c1*h_pref1 * (2.0*U_bar*n2Uphp)

#     summary.loc[key, '<Sf>_T2_h2']           = c2*h_pref2 * ((n_bar**2)*(U_bar**2)*hp2)
#     summary.loc[key, '<Sf>_T2_Uh2']          = c2*h_pref2 * (2.0*(n_bar**2)*U_bar*Uph2)
#     summary.loc[key, '<Sf>_T2_nh2']          = c2*h_pref2 * (2.0*n_bar*(U_bar**2)*nh2)
#     summary.loc[key, '<Sf>_T2_U2h2']         = c2*h_pref2 * ((n_bar**2)*U2h2)
#     summary.loc[key, '<Sf>_T2_nUh2']         = c2*h_pref2 * (4.0*n_bar*U_bar*nUh2)
#     summary.loc[key, '<Sf>_T2_n2h2']         = c2*h_pref2 * ((U_bar**2)*n2h2)

#     summary.loc[key, '<Sf>_T3_h3']           = c3*h_pref3 * ((n_bar**2)*(U_bar**2)*hp3)
#     summary.loc[key, '<Sf>_T3_Uh3']          = c3*h_pref3 * (2.0*(n_bar**2)*U_bar*Uh3)
#     summary.loc[key, '<Sf>_T3_nh3']          = c3*h_pref3 * (2.0*n_bar*(U_bar**2)*nh3)

#     summary.loc[key, '<Sf>_T4_h4']           = c4*h_pref4 * ((n_bar**2)*(U_bar**2)*hp4)

#     # ---- grand total up to quartic order
#     summary.loc[key, '<Sf>_exp4'] = float(Sf_T0 + Sf_T1 + Sf_T2 + Sf_T3 + Sf_T4)

#     # ---- comparison to direct
#     summary.loc[key, '<Sf>_direct_minus_exp4'] = summary.loc[key, '<Sf>_direct'] - summary.loc[key, '<Sf>_exp4']

#     # ===== Stability diagnostics + hybrid estimator =====
#     # Wet mask to avoid infs from h_t ~ 0
#     hmin = 1e-4  # meters; tune to your grid/physics
#     wet  = h_t > hmin

#     # Dimensionless depth fluctuation eps = h'/<h>
#     eps = np.where(wet, hp / h_bar, np.nan)
#     summary.loc[key, 'eps_max'] = float(np.nanmax(np.abs(eps)))
#     summary.loc[key, 'eps_rms'] = float(np.sqrt(np.nanmean(eps**2)))

#     # --- Nondimensional contributions (all share h_bar^{-4/3})
#     ND0 = float(np.nanmean((n**2) * (U_t**2) * (eps**0)))  # <n^2 U^2>
#     ND1 = float(np.nanmean((n**2) * (U_t**2) * (eps**1)))
#     ND2 = float(np.nanmean((n**2) * (U_t**2) * (eps**2)))
#     ND3 = float(np.nanmean((n**2) * (U_t**2) * (eps**3)))
#     ND4 = float(np.nanmean((n**2) * (U_t**2) * (eps**4)))

#     Sf_nd_T0 = h_pref0 * (c0 * ND0)
#     Sf_nd_T1 = h_pref0 * (c1 * ND1)
#     Sf_nd_T2 = h_pref0 * (c2 * ND2)
#     Sf_nd_T3 = h_pref0 * (c3 * ND3)
#     Sf_nd_T4 = h_pref0 * (c4 * ND4)

#     summary.loc[key, '<Sf>_nd_T0'] = Sf_nd_T0
#     summary.loc[key, '<Sf>_nd_T1'] = Sf_nd_T1
#     summary.loc[key, '<Sf>_nd_T2'] = Sf_nd_T2
#     summary.loc[key, '<Sf>_nd_T3'] = Sf_nd_T3
#     summary.loc[key, '<Sf>_nd_T4'] = Sf_nd_T4

#     # Partial sums vs direct (exact)
#     Sf_nd_K0 = Sf_nd_T0
#     Sf_nd_K1 = Sf_nd_K0 + Sf_nd_T1
#     Sf_nd_K2 = Sf_nd_K1 + Sf_nd_T2
#     Sf_nd_K3 = Sf_nd_K2 + Sf_nd_T3
#     Sf_nd_K4 = Sf_nd_K3 + Sf_nd_T4

#     summary.loc[key, '<Sf>_nd_sumK0'] = Sf_nd_K0
#     summary.loc[key, '<Sf>_nd_sumK1'] = Sf_nd_K1
#     summary.loc[key, '<Sf>_nd_sumK2'] = Sf_nd_K2
#     summary.loc[key, '<Sf>_nd_sumK3'] = Sf_nd_K3
#     summary.loc[key, '<Sf>_nd_sumK4'] = Sf_nd_K4

#     summary.loc[key, 'err_nd_K0'] = summary.loc[key, '<Sf>_direct'] - Sf_nd_K0
#     summary.loc[key, 'err_nd_K1'] = summary.loc[key, '<Sf>_direct'] - Sf_nd_K1
#     summary.loc[key, 'err_nd_K2'] = summary.loc[key, '<Sf>_direct'] - Sf_nd_K2
#     summary.loc[key, 'err_nd_K3'] = summary.loc[key, '<Sf>_direct'] - Sf_nd_K3
#     summary.loc[key, 'err_nd_K4'] = summary.loc[key, '<Sf>_direct'] - Sf_nd_K4

#     # High-order dominance metric (dimensionless view)
#     denom = max(1e-12, abs(Sf_nd_T1) + abs(Sf_nd_T2))
#     summary.loc[key, 'nd_T_ratio_34_over_12'] = (abs(Sf_nd_T3) + abs(Sf_nd_T4)) / denom

#     # --- Hybrid exact/series weighting for h^{-4/3}
#     # Series weight in eps; exact weight per-cell (clamped by wet mask)
#     w_series = c0 + c1*eps + c2*(eps**2) + c3*(eps**3) + c4*(eps**4)
#     w_exact  = np.where(wet, (1.0 + eps)**(-4.0/3.0), np.nan)  # = (h_t/h_bar)^(-4/3)

#     # Use series when |eps| <= thresh, else exact
#     thresh = 0.8  # try 0.6–0.9
#     mask_series = np.abs(eps) <= thresh
#     w_hybrid = np.where(mask_series, w_series, w_exact)

#     Sf_hybrid = h_pref0 * float(np.nanmean((n**2) * (U_t**2) * w_hybrid))
#     summary.loc[key, '<Sf>_hybrid'] = Sf_hybrid
#     summary.loc[key, 'hybrid_err']  = summary.loc[key, '<Sf>_direct'] - Sf_hybrid
#     summary.loc[key, 'hybrid_frac_series'] = float(np.nanmean(mask_series))
  

In [ ]:
(summary['<Sf>_hybrid']/summary['<Sf>_direct']).describe()

In [ ]:
summary['hybrid_err'].describe()

In [ ]:
summary[['<Sf>_nd_sumK4', '<Sf>_nd_sumK3', '<Sf>_nd_sumK2', '<Sf>_nd_sumK1']].describe()

In [ ]:
plt.scatter(summary['effect_ratio'], (summary['<Sf>_T0']))

In [ ]:
cols = [c for c in summary.columns if '<Sf>' in c]
cols.sort()

In [ ]:
cols

In [ ]:
summary['<np2>']/summary['<n>2']

In [ ]:
(summary['<Up2>']/summary['<U>']**2).describe()
# summary['<hp2>']/summary['<h>']**2

In [ ]:
summary[cols + ['effect_ratio']].corr().sort_values('effect_ratio')

In [ ]:
summary['r_ratio'] =  summary['alpha_b'] / summary['alpha_v'] 
summary['effect'] =  summary['r_avg'] - summary['r_equiv5'] 
summary['effect_ratio'] =  summary['r_equiv5']/summary['r_avg'] 
summary['effect_ratio_geom'] =  summary['r_equiv5']/summary['r_gmean']


In [ ]:
summary[['<Sf>','<Sf>_exp4', '<Sf>_T0','<Sf>_T1', '<Sf>_T2', '<Sf>_T3', '<Sf>_T4', '<Sf>_approx', '<Sf>_approx2',
         'effect_ratio']].corr()

In [ ]:
summary[['<Sf>_approx', '<Sf>_approx2' , "<Sf>"]].describe()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---------- pick rows & resolve column names ----------
subset = summary.query("hydro_err < 0.05").copy() if "hydro_err" in summary.columns else summary.copy()

direct_col = "<Sf>" if "<Sf>" in subset.columns else ("<Sf>_direct" if "<Sf>_direct" in subset.columns else None)
approx_col = "<Sf>_approx" if "<Sf>_approx" in subset.columns else ("<Sf>_approx2" if "<Sf>_approx2" in subset.columns else None)

if direct_col is None:
    raise KeyError("Need '<Sf>' or '<Sf>_direct' in the dataframe.")
if approx_col is None:
    # build an approximation on-the-fly if components exist
    build_from = ['<Sf>_nbar2_Ubar2','<Sf>_nbar2_Up2','<Sf>_Ubar2_np2','<Sf>_cross_nU','<Sf>_C_Uhp_lin','<Sf>_C_nhp_lin']
    have = [c for c in build_from if c in subset.columns]
    if not have:
        raise KeyError("No '<Sf>_approx' found and no components to build it from.")
    subset['<Sf>_approx'] = subset[have].sum(axis=1)
    approx_col = '<Sf>_approx'   # <-- keep consistent

# which components are present?
component_cols = [
    '<Sf>_nbar2_Ubar2',
    '<Sf>_nbar2_Up2',
    '<Sf>_Ubar2_np2',
    '<Sf>_cross_nU',
    '<Sf>_C_Uhp_lin',
    '<Sf>_C_nhp_lin'
]
component_cols = [c for c in component_cols if c in subset.columns]

# ---------- LaTeX labels (mathtext-safe) ----------
term_labels = {
    '<Sf>_nbar2_Ubar2': r"$\langle h\rangle^{-4/3}\,\langle n\rangle^2\,\langle U\rangle^2$",
    '<Sf>_nbar2_Up2':   r"$\langle h\rangle^{-4/3}\,\langle n\rangle^2\,\langle U'^2\rangle$",
    '<Sf>_Ubar2_np2':   r"$\langle h\rangle^{-4/3}\,\langle U\rangle^2\,\langle n'^2\rangle$",
    '<Sf>_cross_nU':    r"$\langle h\rangle^{-4/3}\,4\,\langle n\rangle\langle U\rangle\,\langle n'U'\rangle$",
    '<Sf>_C_Uhp_lin':   r"$-\frac{8}{3}\,\langle h\rangle^{-7/3}\,\langle n\rangle^2\,\langle U\rangle\,\langle U'h'\rangle$",
    '<Sf>_C_nhp_lin':   r"$-\frac{8}{3}\,\langle h\rangle^{-7/3}\,\langle U\rangle^2\,\langle n\rangle\,\langle n'h'\rangle$",
    approx_col:         r"$\langle S_f\rangle_{\mathrm{approx}}$",
    direct_col:         r"$\langle S_f\rangle$",
}

# ---------- figure ----------
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# swap roles: boxplots on the LEFT (axes[0]), scatter on the RIGHT (axes[1])
ax_box = axes[0]
ax_scatter = axes[1]

plt.subplots_adjust(wspace=0.4, left=0.28, right=0.8)  # extra left for long labels; right leaves room for legend

# ---- Right: scatter of each component vs direct, plus total approx (NO 1:1 line) ----
y = subset[direct_col].to_numpy()
for col in component_cols:
    x = subset[col].to_numpy()
    ax_scatter.scatter(x, y, s=30, alpha=0.6, marker='.', label=term_labels[col])

# total approximation in black
x_tot = subset[approx_col].to_numpy()
ax_scatter.scatter(x_tot, y, s=40, alpha=0.85, marker='.', color='black', label=term_labels[approx_col])

ax_scatter.set_xlabel("Approximation term value")
ax_scatter.set_ylabel(term_labels[direct_col])
ax_scatter.set_title(r"Components and total vs $\langle S_f\rangle$")
ax_scatter.grid(True, alpha=0.3)

# Legend outside on the right of the scatter plot
ax_scatter.legend(
    frameon=False, ncol=1, handletextpad=0.6, columnspacing=1.0,
    loc='center left', bbox_to_anchor=(1.02, 0.5), borderaxespad=0.0
)

# ---- Left: HORIZONTAL boxplots of components + total + direct ----
box_cols   = component_cols + [approx_col, direct_col]
box_data   = [subset[c] for c in box_cols]
box_labels = [term_labels[c] for c in box_cols]

ax_box.boxplot(box_data, labels=box_labels, showmeans=True, vert=False)
ax_box.set_title(r"Distribution of $\langle S_f\rangle$ terms")
ax_box.set_xlabel("Value")
ax_box.set_ylabel("")  # ytick labels are the terms

plt.show()


In [ ]:
summary.query("hydro_err < 0.3")[
    [c for c in summary.columns if 'Sf' in c] + ['effect_ratio']].corr()['effect_ratio'].sort_values()

In [ ]:
summary.query("hydro_err < 0.3")[
    [c for c in summary.columns if 'Sf' in c] ].median().sort_values()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

subset = summary.query("hydro_err < 0.05").copy() if "hydro_err" in summary.columns else summary.copy()

# Columns to plot (leading-order terms)
terms = [
    ('<Sf>_nbar2_Ubar2', r"$\langle h\rangle^{-4/3}\,\langle n\rangle^{2}\,\langle U\rangle^{2}$"),
    ('<Sf>_nbar2_Up2',   r"$\langle h\rangle^{-4/3}\,\langle n\rangle^{2}\,\langle U'^2\rangle$"),
    ('<Sf>_Ubar2_np2',   r"$\langle h\rangle^{-4/3}\,\langle U\rangle^{2}\,\langle n'^2\rangle$"),
    ('<Sf>_cross_nU',    r"$4\,\langle h\rangle^{-4/3}\,\langle n\rangle\langle U\rangle\,\langle n'U'\rangle$")
]

# Resolve direct Sf column
direct_col = '<Sf>' if '<Sf>' in subset.columns else ('<Sf>_direct' if '<Sf>_direct' in subset.columns else None)
if direct_col is None:
    raise KeyError("Need '<Sf>' or '<Sf>_direct' in the dataframe.")

# Make the 4x2 figure
fig, axes = plt.subplots(4, 2, figsize=(14, 14), sharex=False, sharey=False)

last_sc = None
for i, (col, label) in enumerate(terms):
    ax_left  = axes[i, 0]
    ax_right = axes[i, 1]

    if col not in subset.columns:
        # Graceful missing-column message
        for ax, side in [(ax_left, "⟨S_f⟩"), (ax_right, "n_e/⟨n⟩")]:
            ax.text(0.5, 0.5, f"Missing column:\n{col}", ha='center', va='center', fontsize=12)
            ax.set_axis_off()
        continue

    x = subset[col].to_numpy()
    y_direct = subset[direct_col].to_numpy()
    y_ratio  = subset['effect_ratio'].to_numpy() if 'effect_ratio' in subset.columns else np.full_like(x, np.nan)

    # left: term vs <Sf> (direct)
    sL = ax_left.scatter(x=x, y=y_direct,
                         c=subset['r_avg'] if 'r_avg' in subset.columns else None,
                         cmap='GnBu', vmin=0.05, vmax=0.25)
    ax_left.set_xlabel(label)
    ax_left.set_ylabel(r"$\langle S_f\rangle$")
    ax_left.grid(True, alpha=0.3)

    # right: term vs effect_ratio
    sR = ax_right.scatter(x=x, y=y_ratio,
                          c=subset['r_avg'] if 'r_avg' in subset.columns else None,
                          cmap='GnBu', vmin=0.05, vmax=0.25)
    ax_right.set_xlabel(label)
    ax_right.set_ylabel(r"$n_e / \langle n \rangle$")
    ax_right.grid(True, alpha=0.3)

    last_sc = sR  # for colorbar

    # tick density
    from matplotlib.ticker import MaxNLocator
    ax_left.xaxis.set_major_locator(MaxNLocator(nbins=4))
    ax_left.yaxis.set_major_locator(MaxNLocator(nbins=4))
    ax_right.xaxis.set_major_locator(MaxNLocator(nbins=4))
    ax_right.yaxis.set_major_locator(MaxNLocator(nbins=4))

# # Shared colorbar (fallback label if r_avg != <n>)
# if last_sc is not None:
#     cbar = fig.colorbar(last_sc, ax=axes.ravel().tolist(), shrink=0.9, pad=0.02)
#     cbar.set_label(r"$r_{\mathrm{avg}}$")

plt.tight_layout()
# plt.show()


In [ ]:
print ( summary.query("hydro_err < 0.05")[['r_equiv5','<n>', '<na>']].corr().round(2))

# plt.scatter(summary.Cd, summary.r_equiv5, label = power)
# plt.legend()

$$ \frac{\partial \langle U \rangle }{\partial t} + 
% \langle U \rangle  \f]rac{\partial \langle U \rangle  }{\partial x}  + 
\bigg \langle  U' \frac{\partial U' }{\partial x} \bigg \rangle
+ g \bigg \langle  \frac{\partial  h'}{\partial x} \bigg \rangle  + 
g( \langle S_f \rangle - S_o) + \frac{\langle U \rangle  }{\langle h \rangle }\langle p - i \rangle = 0 $$



$$
S_f \approx n^2  \cdot \langle h \rangle^{-4/3} \cdot
(\langle U \rangle^2 + 
\langle U'^2  \rangle)
$$

In [ ]:
plt.scatter(summary['a_gmean'], summary['<a>'])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize = (14, 4))

ax = axes[0]
ax.scatter(x = summary['<Up dUp/dx>'], 
           y = summary['<Sf>'], c = summary.r_avg, cmap = 'GnBu',
        vmin = 0.05, vmax = 0.25)

ax.set_ylabel(r"$\langle S_f \rangle  = \langle U \rangle ^2 \langle n^2\rangle  \langle h \rangle ^{-4/3}$")
ax.set_xlabel(r"$\langle U' dU'/dx \rangle $ ")

# np.nanmean(a**2)*(hhh.mean())**(-4/3)*(vv.mean()**2)   

ax = axes[1]
g = ax.scatter(x = 9.8*summary['<dhp/dx>'], 
               y = summary['<Sf>'], c = summary.r_avg, cmap = 'GnBu',
               vmin = 0.05, vmax = 0.25)
ax.set_xlabel(r"$\langle d h' /dx \rangle $ ")
ax.set_ylabel(r"$\langle S_f \rangle  = \langle U \rangle ^2 \langle n^2\rangle  \langle h \rangle ^{-4/3}$")
         

plt.colorbar(g, label =r"$\langle n \rangle $" )
for ax in axes:
    from matplotlib.ticker import MaxNLocator
    ax.xaxis.set_major_locator(MaxNLocator(nbins=4))  # max 5 ticks
    ax.yaxis.set_major_locator(MaxNLocator(nbins=4))

  

fig, axes = plt.subplots(1, 2, figsize = (14, 4))
ax = axes[0]
ax.scatter(summary['d<U>/dt']/summary['<h>']*(summary['p'] - summary['Ks_v'])/3.6e5,
           summary['<Sf>'], c = summary.r_avg, cmap = 'GnBu',
            vmin = 0.05, vmax = 0.25)
ax.set_xlabel(r"$d \langle U \rangle / dt  \cdot (p - K_s) \cdot \langle h \rangle^{-1}$ ")
ax.set_ylabel(r"$\langle S_f \rangle = \langle U \rangle ^2 \langle n^2\rangle  \langle h \rangle ^{-4/3}$")


ax = axes[1]
g = ax.scatter(summary['<U>/<h>']*(summary['p'] - summary['Ks_v'])/3.6e5, summary['<Sf>'], c = summary.r_avg, cmap = 'GnBu',
            vmin = 0.05, vmax = 0.25)
ax.set_xlabel(r"$\langle U \rangle  \cdot (p - i) \cdot \langle h \rangle^{-1}$ ")
ax.set_ylabel(r"$\langle S_f \rangle = \langle U \rangle ^2 \langle n^2\rangle  \langle h \rangle ^{-4/3}$")
plt.colorbar(g, label =r"$\langle n \rangle $" )
for ax in axes:
    from matplotlib.ticker import MaxNLocator
    ax.xaxis.set_major_locator(MaxNLocator(nbins=4))  # max 5 ticks
    ax.yaxis.set_major_locator(MaxNLocator(nbins=4))


$$
S_f \approx \langle n^2 \rangle  \cdot \langle h \rangle^{-4/3} \cdot
(\langle U \rangle^2 + 
\langle U'^2  \rangle)
$$


In [ ]:
summary.query("tr > 40 and l < 300").groupby('fV').effect_ratio.mean()

In [ ]:

sim.hc[sim.i_tr].min(), (sim.p - sim.Ks_v)/3.6e5*sim['dt']

In [ ]:
summary['<U>^2/<Up^2>'] = (summary['<U>']**2/ summary['<Up2>'])
subset = summary.query("r_avg < 0.1")
subset[['<U>^2/<Up^2>', 'l']].sort_values('<U>^2/<Up^2>')

In [ ]:

fig, axes = plt.subplots(1, 2, figsize = (14, 4), sharex = False)
plt.subplots_adjust(wspace = 0.25)
g = axes[0].scatter(summary['<U>']**2, summary['<Up2>'], 
                    c = summary.r_avg, 
                    cmap = 'GnBu', vmin = 0.05, vmax = 0.25)

axes[0].set_xlabel(r"$\langle U \rangle ^2 $")
axes[0].set_ylabel(r"$\langle U'^2 \rangle $")
plt.colorbar(g, label = r"$\langle n \rangle$" )

ax = axes[1]
g = axes[1].scatter(
                    x = summary['<a2>']/summary['<a>']**2,
                    y = summary['<U>']**2/ summary['<Up2>'],  
                    c = summary.l, 
                    cmap = 'GnBu', vmin = 0, vmax =400)
ax.set_ylabel(r"$\langle U \rangle ^2/ \langle U^2 \rangle $")
ax.set_xlabel(r"$\langle n \rangle^2 / \langle n^2 \rangle $")
plt.colorbar(g, label =r"$\langle n \rangle $" )

for ax in axes:
    from matplotlib.ticker import MaxNLocator
    ax.xaxis.set_major_locator(MaxNLocator(nbins=4))  # max 5 ticks
    ax.yaxis.set_major_locator(MaxNLocator(nbins=4))


In [ ]:

fig, axes = plt.subplots(1, 2, figsize = (14, 4), sharex = True)
g = axes[0].scatter(summary['<U>']**2, summary['<Up2>'], c = summary.r_avg, cmap = 'GnBu', vmin = 0.05, vmax = 0.25)
axes[0].set_xlabel(r"$\langle U \rangle ^2 $")
axes[0].set_ylabel(r"$\langle U'^2 \rangle $")
# plt.colorbar(g, label =r"$\langle n \rangle $" )

ax = axes[1]
g = ax.scatter(summary['<U>']**2,  (summary['<U>']**2- summary['<Up2>'])/summary['<U>']**2,
               c = summary.r_avg, cmap = 'GnBu', vmin = 0.05, vmax = 0.25)
# x = np.linspace(ax.get_xlim()[0], ax.get_xlim()[1], 10)
# ax.plot(x, x)

ax.set_xlabel(r"$\langle U \rangle ^2 $")
ax.set_ylabel(r"$(\langle U \rangle ^2  - \langle U'^2 \rangle)\ / \ \langle U \rangle ^2 $")
plt.colorbar(g, label =r"$\langle n \rangle $" )

for ax in axes:
    from matplotlib.ticker import MaxNLocator
    ax.xaxis.set_major_locator(MaxNLocator(nbins=4))  # max 5 ticks
    ax.yaxis.set_major_locator(MaxNLocator(nbins=4))


In [ ]:
plt.scatter(summary['Sf'], summary.effect_ratio, c = summary.r_avg, cmap = 'GnBu', vmin = 0.05, vmax = 0.25)
plt.xlabel(r"$\langle U  ^2  n^2   h  ^{-4/3} \rangle$")
plt.ylabel(r"$n_e/\langle n \rangle$ ")
plt.colorbar()
plt.xlim(0.0095, 0.0105)


for ax in axes:
    from matplotlib.ticker import MaxNLocator
    ax.xaxis.set_major_locator(MaxNLocator(nbins=4))  # max 5 ticks
    ax.yaxis.set_major_locator(MaxNLocator(nbins=4))


In [ ]:
a = sim.veg
a[a==1] = sim.alpha_v
a[a==0] = sim.alpha_b

(sim.hc[sim.i_tr]**(-4/3)*sim.vc[sim.i_tr]**(2)*a**2).mean()


In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 150
subset = summary.query("hydro_err < 0.05")
# plt.scatter(subset['r_equiv5'], subset['<na>'])
plt.scatter(subset['r_equiv5'], subset['n']/2, label = "$r = {0:.2f}$".format(
    np.corrcoef(subset['r_equiv5'], subset['n'])[0, 1]),)

plt.plot(subset['r_equiv5'], subset['r_equiv5'], 'C3--')
plt.xlabel(r"$n_{eq}$")
plt.ylabel(r"$h(t_r)^{1/6} \langle n \rangle /2$")
plt.legend()
# plt.scatter(subset['r_equiv5'], subset['n']/(5/3.), alpha = 0.5, label ='')

In [ ]:
from scipy.ndimage import gaussian_filter1d
smoothed = gaussian_filter1d(a, sigma=.1, axis=0)

ha = integrate_h_across(smoothed, .66, sim.So, sim.p - sim.Ks_v, dx=2)

plt.plot(ha/sim.l, '.-', label = 'predict $h$')
plt.plot(sim.hc[sim.i_tr, :, :].mean(1), '.-', label = '$h$')
plt.ylim(0,)
plt.legend()

In [ ]:
smoothed = gaussian_filter1d(a, sigma=.1, axis=0)

Ua = integrate_U_across(smoothed, .66, sim.So, sim.p - sim.Ks_v, dx=2)

plt.plot(Ua/sim.l, '.-', label = 'predict $U$')
plt.plot(sim.vc[sim.i_tr, :, :].mean(1), '.-', label = '$U$')
plt.ylim(0,)
plt.legend()

In [ ]:
fig, axes = plt.subplots( 1,3, figsize = (16, 4))
subset = summary.query("tr == 60 ")
# axes[0].plot(subset['<U>_correct'], subset['<U>_correct'])
# axes[0].scatter(subset['<U>_correct'], (subset['<U>_correct']-subset['<U>'])/subset['<U>_correct'],  label = 'lumped')

c = axes[0].scatter(subset['<U>_correct'], (subset['<U>_correct']-subset['<Ua>'])/subset['<U>_correct'], 
                c = subset['sigma'], cmap = 'viridis', label = 'integrate')
# plt.colorbar(c, ax = axes[0])
axes[0].set_xlabel("$U$")
axes[0].set_ylabel("$U$ error")

# axes[1].plot(subset['<h>_correct'], subset['<h>_correct'])
# axes[1].scatter(subset['<h>_correct'], (subset['<h>'] - subset['<h>_correct'])/subset['<h>_correct'],  label = 'lumped')
c = axes[1].scatter(subset['<h>_correct'], (subset['<ha>'] - subset['<h>_correct'])/subset['<h>_correct'], 
                c = subset['sigma'], cmap = 'viridis', label = 'integrate')
plt.colorbar(c, ax = axes[1])
print(np.mean((subset['<ha>'] - subset['<h>_correct'])/subset['<h>_correct']))
axes[1].set_xlabel("$h$")
axes[1].set_ylabel("$h$ error")

axes[2].plot(subset['<h>_correct']/subset['<U>_correct'], subset['<h>_correct']/subset['<U>_correct'])
axes[2].scatter(subset['<h>_correct']/subset['<U>_correct'], subset['<h>']/subset['<U>'])
axes[2].scatter(subset['<h>_correct']/subset['<U>_correct'], subset['<ha>']/subset['<Ua>'], label = 'integrate')
axes[2].legend()
axes[2].set_xlabel("$h/U$")
axes[2].set_ylabel("predicted $h/U$")

# for ax in axes:
#     ax.set_ylim(0, )
#     ax.set_xlim(0, )

# plt.scatter(subset['<V>'], subset['<V>'] - subset['<U>'])
# plt.colorbar()

In [ ]:
print_input_params(sim_list)

In [ ]:
subset[['<hh>', '<ha>', '<h>']].corr().round(4)

In [ ]:
subset[['<UU>', '<Ua>', '<U>']].corr().round(4)

In [ ]:
plot_hydrographs(summary.query("tr == 60 and p > 6 and l == 400"))

In [ ]:
np.corrcoef(subset['<hh>']**(2/3)/subset['<UU>']*subset['a_gmean']**-0.333, subset['r_equiv5'])

np.corrcoef(subset['<ha>']**(2/3)/subset['<Ua>']*subset['a_gmean']**0, subset['r_equiv5'])

In [ ]:
subset[['<n>', 'r_equiv5', 'n']].corr()

In [ ]:
(summary.Cd/summary.Cd.mean()).hist()
plt.xlabel("$C_d/$mean$(C_d)$")
plt.ylabel('count')

In [ ]:


subset = summary.query("hydro_err < {0}".format(0.05))
print (subset.shape)
for key in subset.index:
    sim  = subset.loc[key]        
    a = sim.veg
    a[a==1] = sim.alpha_v
    a[a==0] = sim.alpha_b

    hhh = sim.hc.copy()
    hhh[hhh < 0.002] = np.nan

    #vv = sim.vc.copy()
    #vv[vv < 0.] = np.nan        
    subset.loc[key, 'Cd'] = 9.8*np.nanmean(hhh**(0.166)*a**0.5)
print (subset[['r_equiv', 'Cd']].corr())
plt.scatter(subset.Cd, subset.r_equiv)


In [ ]:
subset = summary.query("hydro_err < 0.05")
fig, axes = plt.subplots(1,2, figsize = (10, 4), sharey = True)
ax = axes[0]
c = ax.scatter(subset.r_gmean, subset.r_equiv5, c = subset.hydro_err, cmap = 'coolwarm')
l = np.arange(ax.get_xlim()[0], ax.get_xlim()[1], 0.01)
axes[0].plot(l, l, c = 'C0', label = '$r_{geom}$')
axes[0].plot(l, l/2, c = 'C2', label = '$r_{geom}/2$')
axes[0].legend()
axes[0].set_xlabel("$r_{geom}$")
axes[0].set_ylabel("$r_{e}$")

plt.colorbar(c,ax = axes[0])

ax = axes[1]
ax.scatter(subset.r_avg, subset.r_equiv5, c = subset.LB, cmap = 'coolwarm')
l = np.arange(ax.get_xlim()[0], ax.get_xlim()[1], 0.01)
axes[1].plot(l, l/2, c = 'C0', label = '$r_{avg}/2$')
axes[1].plot(l, l/3, c = 'C2', label = '$r_{avg}/3$')
axes[1].set_xlabel("$r_{avg}$")
axes[1].set_ylabel("$r_{e}$")
axes[1].legend()

In [ ]:
subset[['r_avg', 'r_equiv5']].corr()

In [ ]:
plt.hist(summary.hydro_err, 20);

In [ ]:
plt.pcolormesh(summary.query(" sigma == 0 ").iloc[0]['veg'])

In [ ]:
plt.pcolormesh(summary.query("l == 400 and sigma == 5 and aniso == 1").iloc[0]['veg'], cmap = "Greens")

In [ ]:
plt.pcolormesh(summary.query("l == 400 and sigma ==5 and aniso == 3").iloc[0]['veg'], cmap = "Greens")

In [ ]:
subset = summary.query("hydro_err < 0.1")
fig, axes = plt.subplots(1,2, figsize = (10, 4), sharey = True)
axes[0].scatter(subset.r_gmean, subset.r_equiv5, c = subset.hydro_err, cmap = 'coolwarm')
l = np.arange(0.04, .11, 0.01)
axes[0].plot(l, l, c = 'C0', label = '$r_{geom}$')
axes[0].plot(l, l/2, c = 'C2', label = '$r_{geom}/2$')
axes[0].legend()
axes[0].set_xlabel("$r_{geom}$")
axes[0].set_ylabel("$r_{e}$")
axes[0].set_ylim(0.0, )
axes[0].set_xlim(0.0, )

axes[1].scatter(subset.r_avg, subset.r_equiv5, c = subset.LB, cmap = 'coolwarm')

l = np.arange(0.04, .2, 0.01)

axes[1].plot(l, l/2, c = 'C0', label = '$r_{avg}/2$')
axes[1].plot(l, l/3, c = 'C2', label = '$r_{avg}/3$')
axes[1].set_xlabel("$r_{avg}$")
axes[1].set_ylabel("$r_{e}$")
axes[1].set_ylim(0.0, )
axes[1].set_xlim(0.0, )
axes[1].legend()

In [ ]:

subset = summary.query("hydro_err < 0.05 and tr == 60")




fig, axes = plt.subplots(1,2, figsize = (10, 4), sharey = True,  sharex = True)
g = axes[0].scatter(subset.Re_all, subset.r_equiv/subset.r_gmean, c = subset.LB, cmap = 'coolwarm')

axes[0].set_xlabel("$Re$")
axes[0].set_ylabel("$r_{e}/r_{geom}$")
plt.colorbar(g)
axes[0].semilogx()

g = axes[1].scatter(subset.Re_all, subset.r_equiv/subset.r_avg, c = subset.LB, alpha = 0.3, cmap = 'coolwarm')

axes[1].set_xlabel("$Re$")
axes[1].set_ylabel("$r_{e}/r_{avg}$")
# axes[0].ylim(0.02, )
# axes[0].xlim(0.02, )
plt.colorbar(g, label = "$L_b$")
axes[1].semilogx()

In [ ]:
fig, axes = plt.subplots(1,2, figsize = (10, 4), sharey = True,  sharex = True)
g = axes[0].scatter(subset.Re_all, subset.r_equiv/subset.r_gmean, c = subset.LB, cmap = 'coolwarm')

axes[0].set_xlabel("$Re$")
axes[0].set_ylabel("$r_{e}/r_{geom}$")
plt.colorbar(g)
axes[0].semilogx()

g = axes[1].scatter(subset.Re_all, subset.r_equiv/subset.r_avg, c = subset.LB, alpha = 0.3, cmap = 'coolwarm')

axes[1].set_xlabel("$Re$")
axes[1].set_ylabel("$r_{e}/r_{avg}$")
# axes[0].ylim(0.02, )
# axes[0].xlim(0.02, )
plt.colorbar(g, label = "$L_b$")
axes[1].semilogx()

In [ ]:
summary.query("hydro_err < 0.05 and IF_err < .05")[['Re_all', 'effect_ratio_geom']].corr('spearman')

## Summary plots

In [ ]:
subset = summary.query("hydro_err < 0.05 and L <= 500")
subset['fV'] = subset['fV'].astype(float).round(3)
plt.scatter(subset.fV, subset.hydro_err, c = subset.l, cmap = 'coolwarm')
plt.xlabel("vegetation fraction")
plt.ylabel("hydrograph error")

In [ ]:
fig, ax = plt.subplots(1, figsize = (10,4))
sim = summary.query(" hydro_err < 0.05 and hydro_err < 0.04 ").iloc[6]

ax.plot(sim.t/60,sim.hydro,   label = "$r_{e}"+"={0:.2f}$".format(sim.r_equiv))
ax.plot(sim.t[:len(sim.equiv_hydro)]/60, sim.equiv_hydro, label = "$r_{avg}$" +"$={0:.2f}$".format(sim.r_avg))
subset= summary_equiv.query("L == {0} and p == 8".format(sim.L))
for key in subset.index[0:]:
    ax.plot(subset.loc[key].t/60,subset.loc[key].hydro, 'g', alpha = 0.1)
ax.legend()
ax.set_xlabel("Time (min)")
ax.set_ylabel("q (cm/hr)")
ax.text(1.5, 0., ' ', transform=ax.transAxes, ha="left", va="top",
            fontsize = 18, style = 'italic');
print (sim.name, sim.hydro_err)

In [ ]:
def plot_q_compare(sim, ax):

    ax.plot(sim.t/60,sim.hydro,  
            label = "$r_{avg}$" +"$={0:.2f}$".format(sim.r_avg))
    ax.plot(sim.t[:len(sim.equiv_hydro)]/60, sim.equiv_hydro, 
             label = "$r_{e}"+"={0:.2f}$".format(sim.r_equiv))
    subset = summary_equiv.query("l == {0}  and p == {1} and tr == {2}".format(
        sim.l, sim.p, sim.tr))
    
    for key in subset.index:
        ax.plot(subset.loc[key].t/60,subset.loc[key].hydro, 'g', alpha = 0.05)
    ax.legend()

    #ax.text(2, 1., ' ', transform=ax.transAxes, ha="left", va="top",
    #            fontsize = 18, style = 'italic');

subset = summary.query("hydro_err < 0.02 and tr == 20").sort_values("l").sample(9)   
fig, axes = plt.subplots(3, 3, figsize = (20,12), sharex = True)
axes = axes.ravel()
for i, ax in enumerate(axes):
    sim = subset.iloc[i]
    plot_q_compare(sim, ax)
    ax.set_xlim(0, sim.tr*3)
    ax.set_title(r'$L$ = {0:.0f}, '.format(sim.l) + '$r_e/r_{avg}$'+ r'={0:.2f}'.format(sim.r_equiv/sim.r_avg))

In [ ]:
subset = summary.query("hydro_err > 0.02 and hydro_err < 0.05").sort_values("l").sample(9)   
fig, axes = plt.subplots(3, 3, figsize = (18,10), sharex = True)
axes = axes.ravel()
for i, ax in enumerate(axes):
    sim = subset.iloc[i]
    plot_q_compare(sim, ax)
    #ax.set_xlim(0, sim.tr*2)
    ax.set_title(r'$L$ = {0:.0f}, '.format(sim.l) + '$r_e/r_{avg}$'+ r'={0:.2f}'.format(sim.r_equiv/sim.r_avg))

In [ ]:

subset = summary.query("hydro_err < 0.02")


rename = {"effect" : "$r_{avg} - r_{e}$",
          'effect_ratio': "$ r_{e}/r_{avg}$", 
          'effect_ratio_geom' : "$ r_{e}/r_{geom}$", 
          'r_avg' : r'$r_{avg}$', 
          'r_gmean' : r'$r_{geom}$', 
          'r_ratio' : '$r_b/r_v$',
          "alpha_b" : r'$r_{b}$',
          'r_equiv': r'$r_e$',
          'l' : '$L$',
          'seed' : 'seed',
          'sigma' : r'$\sigma$',
          'radius' : 'r',
          'fV' : '$F_v$',
          'z_std' : r"$\sigma(z)$", 
          'Re_all' : '$Re$',
          'r_avg_off' : r'$r_{avg}$',
          'tr' :'$t_R$',
          'p' : '$p$',
          'Re' : '$Re$'
         }
def renameit (name, rename = rename):
    try: 
        return rename[name]
    except:
        return name
    
yfld = "effect_ratio_geom"
xfld  = 'sigma'
cfld = 'aniso'
sfld = 'l'
query ='hydro_err < 0.05 and tr == 60 and  p == 8'
    
plt.figure(figsize = (6, 4))
subset = summary.query(query).copy()
subset.loc[subset.query("tr == 60").index, 'l'] =subset.loc[subset.query("tr == 60").index, 'l'] + 10
subset.loc[subset.query("tr == 60").index, 'sigma'] =subset.loc[subset.query("tr == 60").index, 'sigma'] + .2
# subset.loc[subset.query("tr == 60").index, 'aniso'] =subset.loc[subset.query("tr == 60").index, 'aniso'] + .2
subset = subset.rename(rename, axis = 1)
g = sns.boxplot(data = subset,
                    x = renameit(xfld), 
                    y = renameit(yfld), 
#                     size = renameit(sfld),  
                    hue = renameit(cfld),
                   dodge = True)
                   
g.legend(loc='center left', bbox_to_anchor=(1.0, 0.5), ncol=1)    

# the effect size increases with 

In [ ]:
subset = summary.query(query).copy()
summary.groupby("aniso")['effect_ratio'].mean()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize = (16, 4), sharey = True)
for i, l in enumerate(summary.l.unique()):
    subset = summary.query("hydro_err < 0.05 and tr == 60 and l == {0} and p == 8 and fV > 0.15".format(l)).copy()
    subset.loc[subset.query("aniso == 3").index, 'sigma'] += .25
    sns.scatterplot(subset,
                x = 'sigma', ax = axes[i], palette = ['C0', 'C2'],
                y = 'effect_ratio_geom', hue = 'aniso', s = 100)
    axes[i].set_title("hillslope $L$ = {0:.0f}".format(l))
    axes[0].set_ylabel(renameit('effect_ratio_geom'))    
    x = plt.suptitle("duration = 60 minute, p = 8 cm/hr, veg fraction = 0.2", y = 1.03)
    
    
fig, axes = plt.subplots(1, 4, figsize = (16, 4), sharey = True)
for i, l in enumerate(summary.l.unique()):
    subset = summary.query("hydro_err < 0.05 and tr == 60 and l == {0} and p == 8 and fV < 0.15".format(l)).copy()
    subset.loc[subset.query("aniso == 3").index, 'sigma'] += .25
    sns.scatterplot(subset,
                x = 'sigma', ax = axes[i], palette = ['C0', 'C2'],
                y = 'effect_ratio_geom', hue = 'aniso',  s= 100)
    axes[i].set_title("hillslope $L$ = {0:.0f}".format(l))    
    axes[0].set_ylabel(renameit('effect_ratio_geom'))    
    x = plt.suptitle("duration = 60 minute, p = 8 cm/hr, veg fraction = 0.1", y = 1.03)
    
fig, axes = plt.subplots(1, 4, figsize = (16, 4), sharey = True)
for i, l in enumerate(summary.l.unique()):
    subset = summary.query("hydro_err < 0.05 and tr == 20 and l == {0} and p == 8".format(l)).copy()
    subset.loc[subset.query("aniso == 3").index, 'sigma'] += .25
    sns.scatterplot(subset,
                x = 'sigma', ax = axes[i], palette = ['C0', 'C2'],
                y = 'effect_ratio_geom', hue = 'aniso',  s= 100)
    axes[i].set_title("hillslope $L$ = {0:.0f}".format(l))
    axes[0].set_ylabel(renameit('effect_ratio_geom'))
    x = plt.suptitle("duration = 20 minute, p = 8 cm/hr, veg fraction = 0.2", y = 1.03)  
    # results get better with increasing scale, this might be a good thing – physics operate bigger scale.s
    #  subgrid heterogeneity.
    #  advances in remote sensing provide later hs on larger scale incompatible with RS, need to coarse grain spatial averaging, need to coarse grain the physics.
    
    

In [ ]:
print_input_params(sim_list)

In [ ]:
dUdx = (sim.vc[sim.i_tr,:, 2:] - sim.vc[sim.i_tr,:, :-2])/2/sim.dx
U_dUdx = sim.vc[sim.i_tr, :, 1:-1]*dUdx

In [ ]:
fig, ax = plt.subplots(1, figsize = (10,4))
# subset = summary.query("hydro_err < 0.05 and p == 8 and tr == 60 and l == 200 and fV < 0.15 ")
# plot_hydrographs(subset,ax = ax)

subset = summary.query("hydro_err < 0.05 and p == 8 and tr == 60 and l == 200 and fV > 0.15 and effect_ratio_geom >.9 ")
plot_hydrographs(subset, c= 'C1', ax =ax)

subset = summary.query("hydro_err < 0.05 and p == 8 and tr == 60 and l == 200 and fV > 0.15 and effect_ratio_geom <0.8 ")
plot_hydrographs(subset, c= 'C2', ax =ax)



fig, ax = plt.subplots(1, figsize = (10,4))
# subset = summary.query("hydro_err < 0.05 and p == 8 and tr == 60 and l == 200 and fV < 0.15 ")
# plot_hydrographs(subset,ax = ax)

subset = summary.query("hydro_err < 0.05 and p == 8 and tr == 60 and l == 200 and fV > 0.15 and sigma >3  ")
plot_hydrographs(subset, c= 'C1', ax =ax)

subset = summary.query("hydro_err < 0.05 and p == 8 and tr == 60 and l == 200 and fV > 0.15 and sigma <3 ")
plot_hydrographs(subset, c= 'C2', ax =ax)


In [ ]:
query ='hydro_err < 0.05 and tr == 60 and p ==8 and fV < 0.15'
cfld = 'l'
sfld = 'aniso'
yfld = "effect_ratio_geom"
xfld  = 'sigma'
plt.figure(figsize = (6, 4))

subset = summary.query(query).copy()
for i, s in enumerate(subset.l.unique()):
    subset.loc[subset.query("l == {0}".format(s)).index, 'sigma'] = subset.loc[subset.query("l == {0}".format(s)).index, 'sigma'] + i/8
subset = subset.rename(rename, axis = 1)
g = sns.scatterplot(data = subset,
                    x = renameit(xfld), 
                    y = renameit(yfld), 
                    size = renameit(sfld),  
                    hue = renameit(cfld), palette = "coolwarm"
                   )
g.legend(loc='center left', bbox_to_anchor=(1.0, 0.5), ncol=1)    
plt.title("$t_r = 60$ min")

In [ ]:
sim.veg.mean()

In [ ]:
summary.fV.describe()

In [ ]:
query ='hydro_err < 0.03 and tr == 20 and p == 8 '
cfld = 'sigma'
sfld = 'l'
plt.figure(figsize = (6, 4))
xfld = 'r_avg'
yfld = 'effect_ratio'
subset = summary.query(query).copy()
for s in subset.sigma.unique():
    subset.loc[subset.query("sigma == {0}".format(s)).index, 'l'] =subset.loc[subset.query("sigma == {0}".format(s)).index, 'l'] + s*5
subset = subset.rename(rename, axis = 1)
g = sns.scatterplot(data = subset,
                    x = renameit(xfld), 
                    y = renameit(yfld), alpha = 0.8,
                    size = renameit(sfld),  
                    hue = renameit(cfld), palette = "coolwarm"
                   )



plt.title("$t_r = 20$ min, $p$ = 8 cm/hr, maximum 2% error")

subset = summary.query(query).copy()

g.legend(loc='center left', bbox_to_anchor=(1.0, 0.5), ncol=1)    

In [ ]:
query ='hydro_err < 0.05 and l == 400 '
cfld = 'sigma'
sfld = 'sigma'
xfld = 'r_avg'
yfld = 'r_equiv'
plt.figure(figsize = (6, 4))
subset = summary.query(query).copy()
subset.loc[subset.query("sigma == 2").index, 'l'] =subset.loc[subset.query("sigma == 2").index, 'l'] + 10
subset = subset.rename(rename, axis = 1)
g = sns.scatterplot(data = subset,
                    x = renameit(xfld), 
                    y = renameit(yfld), 
                    size = renameit(sfld),  
                    hue = renameit(cfld), palette = "coolwarm"
                   )
g.legend(loc='center left', bbox_to_anchor=(1.0, 0.5), ncol=1)    

plt.title("$t_r = 40$ min")
ax = plt.gca()
ax.set_ylim(0,)
ax.set_xlim(0,)

In [ ]:
query = 'hydro_err < 0.04 and fV > 0.16 and fV < 0.19 and p == 5 and  l == 400 and tr == 60'
fig, axes = plt.subplots(2, 2, figsize = (16,8))
axes = axes.ravel()
sim = summary.query(query).query("effect_ratio < 0.45 ").iloc[0]
axes[0].imshow(sim.veg)

axes[0].set_title("roughness ratio = {0:.2f}, $f_V$ = {1:.2f}".format(sim.effect_ratio, sim.veg.mean()))
plot_q_compare(sim, axes[1])

sim = summary.query(query).query("effect_ratio > 0.42 ").iloc[0]
axes[2].imshow(sim.veg)
axes[2].set_title("roughness ratio = {0:.2f}, $f_V$ = {1:.2f}".format(sim.effect_ratio, sim.veg.mean()))


plot_q_compare(sim, axes[3])
# the effect size increases with 

In [ ]:
plt.figure(figsize = (7,4))
subset = summary.query("hydro_err < 0.05 and tr == 60 and p == 5 and l == 400")
plt.scatter(subset.fV, subset.effect_ratio, label = "$t_R$ = 40 min, p = 5 cm/hr")

subset = summary.query("hydro_err < 0.05 and tr == 60 and p == 8 and l == 400")
plt.scatter(subset.fV, subset.effect_ratio, label = "$t_R$ = 40 min, p = 8 cm/hr")
plt.legend()
plt.xlabel("$r_{avg}$")
plt.ylabel("$r_e/r_{avg}$")

In [ ]:
query = 'hydro_err < 0.04 and effect_ratio > 0.42  and p == 5 and  l == 400 and tr == 60'
fig, axes = plt.subplots(2, 2, figsize = (16,8))
axes = axes.ravel()
sim = summary.query(query).query("fV < 0.14 ").iloc[0]
axes[0].imshow(sim.veg)

axes[0].set_title("roughness ratio = {0:.2f}, $f_V$ = {1:.2f}".format(sim.effect_ratio, sim.veg.mean()))
plot_q_compare(sim, axes[1])

sim = summary.query(query).query("fV > 0.16 ").iloc[0]
axes[2].imshow(sim.veg)
axes[2].set_title("roughness ratio = {0:.2f}, $f_V$ = {1:.2f}".format(sim.effect_ratio, sim.veg.mean()))


plot_q_compare(sim, axes[3])
# the effect size increases with 

In [ ]:
query = 'hydro_err < 0.02'
fig, axes = plt.subplots(2, 2, figsize = (16,6))
axes = axes.ravel()
sim = summary.query(query).query(" sigma == 1 and aniso == 1 and l == 400 and tr == 60  and fV > 0.15").iloc[-1]
axes[0].imshow(sim.veg, cmap = 'Greens')

axes[0].set_title("$r_e/r_{geom}$" +" = {0:.2f}, $f_V$ = {1:.2f}".format(sim.effect_ratio_geom, sim.veg.mean()))
plot_q_compare(sim, axes[1])

sim = summary.query(query).query("sigma == 1 and aniso == 3 and l == 400 and tr == 60 and fV > 0.15").iloc[-1]
axes[2].imshow(sim.veg, cmap = 'Greens')

axes[2].set_title("$r_e/r_{geom}$ " + "= {0:.2f}, $f_V$ = {1:.2f}".format(sim.effect_ratio_geom, sim.veg.mean()))


plot_q_compare(sim, axes[3])
# the effect size increases with 

In [ ]:
summary.effect_ratio.max()

In [ ]:
yfld = "effect_ratio_geom"
xfld  = 'Re'
cfld = 'l'
sfld = 'sigma'

df = summary.query("hydro_err < 0.05 and tr == 60 and p == 8").rename(rename, axis = 1)
df = df.loc[:,~df.columns.duplicated()]

g = sns.scatterplot(data = df,
                    x = renameit(xfld), 
                    y = renameit(yfld), 
                    size = renameit(sfld),  
                    hue = renameit(cfld), palette = mako(len(summary.query(query)[cfld].unique()))
                   )
g.legend(loc='center left', bbox_to_anchor=(1.0, 0.5), ncol=1)    
plt.semilogx()

In [ ]:
xfld = "r_gmean"
yfld  = 'effect_ratio'
sfld = "sigma"
cfld = 'p'

query ='hydro_err < 0.04 and Re_all > 50'
g = sns.scatterplot(data = summary.query(query).rename(rename, axis = 1), 
                    x = rename[xfld], y = rename[yfld],  size= rename[sfld], hue = rename[cfld], 
                    palette = mako(len(summary.query(query)[cfld].unique()))
                   )


g.legend(loc='center left',  bbox_to_anchor=(1.0, 0.5), ncol=1)    

In [ ]:
yfld = "effect_ratio_geom"
xfld  = 'tr'
sfld = "p"
cfld = 'p'

query ='hydro_err < 0.04'
g = sns.boxplot(data = summary.query(query).rename(rename, axis = 1), 
                    x = rename[xfld], y = rename[yfld],  
#                 size= rename[sfld], 
                hue = rename[cfld], 
                    palette = mako(len(summary.query(query)[cfld].unique()))
                   )

g.legend(loc='center left',  bbox_to_anchor=(1.0, 0.5), ncol=1)    


In [ ]:
yfld = "effect_ratio"
xfld  = 'r_avg'
cfld = "r_ratio"
sfld = 'l'
g = sns.scatterplot(data = summary.query(query).rename(rename, axis = 1), 
                    x = rename[xfld], y = rename[yfld],  size= rename[sfld], hue = rename[cfld], 
                    palette = mako(len(summary.query(query)[cfld].unique()))
                   )
g.legend(loc='center left',  bbox_to_anchor=(1.0, 0.5), ncol=1)    

In [ ]:
# fit a power law? 
#  run around effect is not particularly scale dependent
#  use tracers to quantify the run around

In [ ]:
plt.figure(figsize = (7, 3))


yfld = "effect_ratio_geom"
xfld  = 'Re_all'
cfld = "tr"
query ='hydro_err < 0.04 and Re_all > 100'
df = summary.query(query).rename(rename, axis = 1)
df = df.loc[:,~df.columns.duplicated()]

g = sns.scatterplot(data = df,
                    x = rename[xfld], y = rename[yfld],  size= '$L$', hue = rename[cfld], 
                    palette =  mako(len(summary.query(query)[cfld].unique()))
                   )
g.legend(loc='center left',  bbox_to_anchor=(1.0, 0.5), ncol=1)    


In [ ]:
plt.figure(figsize = (7, 3))


yfld = "effect_ratio_geom"
xfld  = 'Re_all'
cfld = "tr"
query ='hydro_err < 0.04 and Re_all < 100'
df = summary.query(query).rename(rename, axis = 1)
df = df.loc[:,~df.columns.duplicated()]

g = sns.scatterplot(data = df,
                    x = rename[xfld], y = rename[yfld],  size= '$L$', hue = rename[cfld], 
                    palette =  mako(len(summary.query(query)[cfld].unique()))
                   )
g.legend(loc='center left',  bbox_to_anchor=(1.0, 0.5), ncol=1)    


In [ ]:
# perhaps compare to giraldez + woolhiser – time to steady state?
fig, axes = plt.subplots(1, 2, figsize = (12, 3))
plot_hydrographs(summary.query("l == 400"), ax= axes[0])
plot_hydrographs(summary_equiv.query("l == 400 and tr == 60"), ax = axes[1])


In [ ]:
fig, axes = plt.subplots(1,6,figsize = (13, 3) )
axes = axes.ravel()
for i, key in enumerate(summary.query("L > 20 and L < 500").index[:24][::4]):
    sim = summary.loc[key]
    axes[ i].imshow(sim.veg.T, cmap = "Greens")
    axes[i].imshow(sim.veg.T, cmap = "Greens")

    bin_max = np.maximum(sim.LB_dist.max(), sim.L_dc_dist.max())
 
    axes[i].set_xticks([])
    axes[i].set_yticks([])
    axes[i].set_title("$L_v$ = {0:.1f}".format(sim.LV))
    

## Add tracers

In [ ]:
# reload modules
my_modules = ['plot_SWOF', "read_SWOF", "write_SWOF", "plot_config", 
              "topo", "run_around",
              "source_functions_1p3"]

for mod in my_modules :
    if mod in sys.modules: 
        del sys.modules[mod] 

from plot_SWOF import *
from read_SWOF import *
from plot_config import *
from topo import *
from write_SWOF import *

from run_around import *
from source_functions_1p3 import *

In [ ]:
%%time
from tqdm import tqdm
from multiprocess import Pool


tracer_cols = [ 'y_f', 'y_i', 'x_f', 'x_i', 't_f', 't_i', 'veg_bin',
        'std_x',  't_esc', 'residence', 'length','escape', 'escape_in_domain', 
        'time_to_escape', 'time_to_infiltrate', 'distance']

# params = {"h_min" : 1e-4, "y_min" : 1, "t_min" : 1, 'N' : 2000, "initialize" : 'top'}
params = {"h_min" : 5e-3, "y_min" : 1,  "t_min" : 1, 'N' : 1000}
tracer_name = ",".join(["{0}-{1}".format(key, val) for key,val in params.items() ])
print (tracer_name)

def parallel_tracers(key):

    sim = summary.loc[key]  

    if sim.Tbound == 4:
        periodic = 1
    else: 
        periodic = 0
        
    if 't_scale' not in params.keys():
        params.update({"t_scale" : 1})
        t_scale = 1

        
    if 'initialize' not in params.keys():
        params.update({"initialize" : False})
        initialize = False
    
    if sim.Tbound == 4:
        periodic = 1
    else: 
        periodic = 0
            
    positions, recap = integrate_positions(sim, N = params['N'],  initialize = params['initialize'], 
        t_min = params['t_min'],  y_min = params['y_min'], h_min = params['h_min'], 
        t_scale = params['t_scale'], periodic = periodic)
    
    for key, item in params.items():
        
        recap[key]= item

    for fld in tracer_cols: 
        recap[fld] = positions[fld]    

    return recap

run_tracers = 1

if run_tracers:
    
    max_pool = 8
    
    with Pool(max_pool) as p:
        pool_outputs = list(
            tqdm(
                p.imap(parallel_tracers,
                      summary.index),
                total=len(summary)
            )
        )    

    paired = pd.concat(pool_outputs)
    print(paired.shape)

    tracers = pd.DataFrame()
    tracer_cols = pool_outputs[0].index
    for fld in tracer_cols:
        tracers = insert_fld(tracers,fld)

    for i, key in enumerate(summary.index):
        tracers.loc[key] = pool_outputs[i]
        
    tracers.to_pickle(os.path.join(out_dir, tracer_name + ".pkl"))

    shutil.copy("/Users/octaviacrompton/Projects/roughness-scale/swof_code/source_functions.py", out_dir  + "/code/" + tracer_name + "_source.py" );        
    print ("saving: " + tracer_name)
else:
    tracers = pd.read_pickle(os.path.join(out_dir, tracer_name + ".pkl"))
    print ("reading: " +  tracer_name)

In [ ]:
save_cols = ['scenario', 'r', 'Re_all', 'tr', 'p', 'fV', 'Ks_v', 
        'sigma', 'phi_veg', 'l', 'L',  'C', 'infl_frac']

summary[[col for col in save_cols if col in summary]].to_pickle(
    out_dir + "/summary_abrev.pkl")    

# tracers = pd.read_pickle(out_dir + "/" + tracer_name + ".pkl")

try:
    summary = delete_tracers(summary, tracers)
    summary = add_tracers(summary, tracers)

    
except:
    summary = add_tracers(summary, tracers)


## tracer validation

In [ ]:
cfld = 'curve'
subset = summary.query("L > 0")
plt.scatter(1- subset.infl_frac, (subset.tracer_err  ), c = subset[cfld], 
            cmap = "coolwarm")

plt.xlabel("$C$");
plt.ylabel("$C$ error");

plt.colorbar(label = format_name(cfld));

In [ ]:
cfld = 'l'
subset = summary
plt.scatter(subset.IF, (subset.tracer_IF), c = subset[cfld], 
            cmap = "coolwarm")

plt.plot(subset.IF, (subset.IF))
plt.xlabel("$IF$");
plt.ylabel("tracer $IF$");

plt.colorbar(label = format_name(cfld));

## New heading

In [ ]:
def format_plot(summary, xfld, yfld, cfld, cmap = cm.ocean, 
                alpha = 0.5, err_tol = 0.1, vmax = None, vmin = None,
                s = None, edgecolor = None):
    '''
    Will create a color-coded scatter plot for tracer fields
    '''
    if err_tol:
        inds = summary.query('abs_err < {0}'.format(err_tol)).index
    else: 
        inds = summary.index
    c = plt.scatter(summary.loc[inds][xfld].astype(float), 
                summary.loc[inds][yfld].astype(float), 
                c = summary.loc[inds][cfld].astype(float),  cmap = cmap, visible = 0, vmin = vmin, vmax = vmax)

    plt.scatter(summary.loc[inds][xfld].astype(float),  
                summary.loc[inds][yfld].astype(float),  
                c = summary.loc[inds][cfld].astype(float),  cmap = cmap, alpha =alpha,
                vmin = vmin, edgecolor = edgecolor,
                 vmax = vmax, s = s)

    plt.xlabel(format_name(xfld))
    plt.ylabel(format_name(yfld))
    plt.colorbar(c ,label = format_name(cfld))



In [ ]:
summary.groupby([ "tr", "l"])[['effect_ratio_geom', 'curve']].corr().iloc[::2][['curve']].round(3)

In [ ]:
plt.figure(figsize = (6,4))
cfld = "tr"
xfld = "curve"
yfld = 'effect_ratio_geom'
subset = summary.query("hydro_err < 0.05 and fV >  0.15 and curve > 0.01 and l == 50")
format_plot(subset, xfld, yfld, cfld, cmap = MAKO, alpha = 1)
plt.ylabel(rename[yfld])
plt.gca().set_xscale('log')
subset[['effect_ratio', 'curve']].corr().iloc[0, 1]

In [ ]:
# tortuosity here looks really low – visuals from GRL paper?
cfld = "LB"
xfld = "curve"
yfld = 'effect_ratio_geom'
subset = summary.query("hydro_err < 0.05 and fV > 0.15 and abs_err < 0.05 and curve > 0.01 and tr == 60")
format_plot(subset, xfld, yfld, cfld, cmap = MAKO, alpha = 1)
plt.ylabel(rename[yfld])
plt.gca().set_xscale('log')
subset[['effect_ratio', 'curve']].corr().iloc[0, 1]

In [ ]:
# tortuosity here looks really low – visuals from GRL paper?
xfld = "LV"
cfld = "fV"
yfld = 'effect_ratio_geom'
subset = summary.query("hydro_err < 0.05 and l > 10 and abs_err < 0.05")
format_plot(subset, xfld, yfld, cfld, cmap = MAKO, alpha = 1)
plt.ylabel(rename[yfld])
# plt.gca().set_xscale('log')
subset[['effect_ratio', 'curve']].corr().iloc[0, 1]

In [ ]:
plt.scatter(summary.l, [l.std() for l in summary.LB_dist], c = summary.sigma, cmap = 'coolwarm')
plt.xlabel("hillslope length")
plt.ylabel(" $L_b \cdot F_v$")
# roughness 

In [ ]:
summary['LB_std'] = [l.std() for l in summary.LB_dist]

In [ ]:
updates = {"effect_ratio" : "r_e/r_{avg}", 
           'Delta_I_v' : '\\Delta IF_V',
           'Ks_v' : 'K_s',
           'K_avg' : ' K_{avg}',
           'I_trend' : 'dI/dt',
           'I_trend/Ks' : 'K_s^{-1} dI/dt',           
          }

def format_name(fld, updates = updates):
    names.update(updates)

    if fld in names:
        return  '${0}$'.format(names[fld]).replace("\\\\", "\\").replace("D_", "\Delta \ ").replace("_cal", "_c")
    else: 
        return '${0}$'.format(fld).replace("\\\\", "\\").replace("D_", "\Delta ").replace("_cal", "_c")


       
def nrmse(x,y):
    
    return np.sqrt(np.sum((x - y)**2)/np.size(x))/np.mean(x)

def dislay_fit(subset, ax, target = 'f_cal',  cols = ['fV', 'l'], 
               cfld = "fV", colorbar = False):

    subset[cols] += 1

    res, subset, equation = wrap_fit(subset, cols , target )
    nrmse_val = nrmse(subset[target], make_prediction(res, cols, subset))
    
    c = ax.scatter( 
                 (subset[target]), 
                 (make_prediction(res, cols, subset)),
                 c = subset[cfld], alpha = 0.8, label = '$R^2 = {0:.2f}$'.format(res.rsquared, nrmse_val),
                 cmap = "coolwarm")

    ax.legend(loc = 'upper left', handletextpad=0.1)
    if colorbar == True:
        cbaxes = fig.add_axes([0.91, 0.1, 0.008, 0.75])
        plt.colorbar(c, label = format_name(cfld), cax  = cbaxes)

    ax.set_xlabel(format_name(target).replace("m1", "T"));
    ax.set_ylabel( 'predicted ' + format_name(target).replace("m1", "T"));

    x = np.linspace(*ax.get_xlim())
    ax.plot(x, x, c = 'k', lw = .5)


    ax.set_ylim(np.percentile(subset[target], 0)/2, )
    ax.set_xlim(np.percentile(subset[target], 0)/2, )

    return ax, equation, res

plt.figure(figsize = (6,4))

ax, equation, res = dislay_fit(summary.query("hydro_err < 0.1"), ax = plt.gca(),
                               target = 'r_equiv',  cfld = 'fV',
                               cols = [  'fV',   'p', 'tr', 'l', 'aniso',  'sigma',  'curve',  'LV', 
#                                        'Re_all', 
#                                        'LB',
                                      ])

plt.xlabel("$r_e$")
plt.ylabel("$\hat r_e$")

plt.title (equation);

In [ ]:
ax, equation, res = dislay_fit(summary.query("hydro_err < 0.1"), ax = plt.gca(), 
                               target = 'effect_ratio',  cfld = 'fV',
                                cols = [  'fV',   'p', 'tr', 'l', 'aniso',  'sigma',  'curve',  'LV'
                                      ])


In [ ]:

ax, equation, res = dislay_fit(summary, ax = plt.gca(), target = 'effect_ratio', 
                               cols = ['Re_all', 'fV', 'LB'])

In [ ]:
summary.groupby("l")[['curve', 'r_equiv']].corr().reset_index().drop(
    ['level_1', 'curve'], axis = 1).drop([1,3,5,7]).round(2)

In [ ]:
summary.query("abs_err < 0.1").groupby("l")[['curve', 'effect_ratio']].corr().reset_index().drop(
    ['level_1', 'curve'], axis = 1).drop([1,3,5,7])

In [ ]:
def plot_3D_trajectories(sim, positions, escape = '', ax = '', trapped = '', title = '', dcut = 1,  
                         ucut = 70, min_curve = 0., point = 1, stack = 0, alpha = 0.1):
    '''
    '''
    if dcut == 0:
        dcut = 1
        
    if ax == '':
        fig = plt.figure(figsize= (6,5))
        ax =    fig.add_subplot(111, projection='3d');

    ax = plot_surface(sim, 'veg', title, ucut = ucut, dcut = dcut, 
                      alpha = 0.5,vmax = 1.5, color = cm.Greens,
                      plot_veg = 0, ax = ax, stack = stack);

    for ind in range(len(positions)):

        p = positions.iloc[ind]
        xo = np.array(p.xo)
        yo = np.array(p.yo)    
        xo = xo[yo <= sim.l-dcut*sim.dx] 
        yo = yo[yo <= sim.l-dcut*sim.dx] 
        
        if len(xo) > 0 and p.escape == 1:
            xo = np.concatenate((xo , [xo[-1]]))
            yo = np.concatenate((yo , [sim.l-dcut*sim.dx]))
        
        xo = xo[yo >= ucut*sim.dx]
        yo = yo[yo >= ucut*sim.dx]
        zo = np.array([sim.zc[int(xo[i]/sim.dx), int(yo[i]/sim.dx)] for i in range(len(xo))])
        
        if escape != '':

            if p.escape == 1 and np.std(xo) > min_curve:
                ax.plot(xo,  yo, zo, escape, alpha = alpha) 

        if trapped != '':

            if p.escape == 0 and np.std(xo) > min_curve:
                ax.plot(xo,  yo, zo, trapped, alpha = alpha)  ;    
                
                if point == 1:
                    ax.plot(xo[-1],  yo[-1], zo[-1], trapped +  '.', ms = 5, alpha = 0.4) ;                         

    # ax.set_xlim(ucut*sim.dx, sim.l-dcut*sim.dx)
    ax.view_init(25, 20);

    return ax


In [ ]:
summary.curve.max()

In [ ]:
sim = summary.query("curve > 0.35").iloc[1]

In [ ]:
# reload modules
my_modules = ['plot_SWOF', "read_SWOF", "write_SWOF", "plot_config", 
              "topo", "run_around",
              "source_functions_1p3"]

for mod in my_modules :
    if mod in sys.modules: 
        del sys.modules[mod] 

from plot_SWOF import *
from read_SWOF import *
from plot_config import *
from topo import *
from write_SWOF import *

from run_around import *
from source_functions_1p3 import *

In [ ]:
positions, recap = integrate_positions(sim, N = 500)

In [ ]:
plot_3D_trajectories(sim, positions, ax = '', 
                     escape = 'b', trapped = 'b', alpha = 0.2,
                     min_curve = 0.0, point = 0, 
                     ucut = 1, dcut = 0);
